In [1]:
import sys
from pathlib import Path
import pandas as pd
import json

#finding repository root
cwd = Path.cwd()

if (cwd / "agents").exists():
    repo_root = cwd
elif (cwd.parent / "agents").exists():
    repo_root = cwd.parent
else:
    raise FileNotFoundError("Cannot locate repository root.")

#accessing metadata agent
sys.path.insert(0, str(repo_root))

from agents.metadata_agent import (
    extract_metadata,
    sql_profile,
    primary_keys,
    possible_relationships,
    export_metadata
)

#imaging dataset path
image_path = repo_root / "datasets" / "ventricular data"

assert image_path.exists(), "Imaging dataset directory not found."

print("Repository:", repo_root)
print("Imaging dataset:", image_path)

Repository: /Users/rakshanpatnaik/Desktop/D37-Semantic-Data-Lake
Imaging dataset: /Users/rakshanpatnaik/Desktop/D37-Semantic-Data-Lake/datasets/ventricular data


### Checking overall metadata

In [2]:
# datasets
heart_path = repo_root / "datasets" / "cleaned" / "heart_disease_cleaned.csv"
cardio_path = repo_root / "datasets" / "cleaned" / "cardiovascular_cleaned.csv"

heart = extract_metadata(heart_path)
cardio = extract_metadata(cardio_path)

print(heart["row_count"], heart["column_count"])
print(cardio["row_count"], cardio["column_count"])

297 16
1000 15


In [3]:
# testing overall metadata
print(heart["row_count"], heart["column_count"])
print(cardio["row_count"], cardio["column_count"])

297 16
1000 15


### Checking column metadata

In [4]:
# testing for trestbps column
heart["columns"]["trestbps"]

{'datatype': 'int64',
 'null_count': 0,
 'null_percentage': 0.0,
 'all_missing': False,
 'distinct_count': 50,
 'column_sample': [145, 160, 120, 130, 130],
 'statistics': {'min': 94.0,
  'max': 200.0,
  'mean': 131.69360269360268,
  'median': 130.0},
 'sql_profile': {'null_count': 0,
  'distinct_count': 50,
  'statistics': {'min': 94,
   'max': 200,
   'mean': 131.69360269360268,
   'median': 130.0}}}

In [5]:
# testing for restingBP column
cardio["columns"]["restingBP"]

{'datatype': 'int64',
 'null_count': 0,
 'null_percentage': 0.0,
 'all_missing': False,
 'distinct_count': 95,
 'column_sample': [171, 94, 133, 138, 199],
 'statistics': {'min': 94.0, 'max': 200.0, 'mean': 151.747, 'median': 147.0},
 'sql_profile': {'null_count': 0,
  'distinct_count': 95,
  'statistics': {'min': 94, 'max': 200, 'mean': 151.747, 'median': 147.0}}}

In [6]:
# note: in these datasets, there are no categorical-encoded categorical datasets.

### Validation with EDA

In [7]:
# checking for number of missing values across datasets according to EDA
print(sum(info["null_count"] for info in heart["columns"].values())) # should be 0
print(sum(info["null_count"] for info in cardio["columns"].values())) # should be 53

0
53


### Checking profiling summary statistics

In [8]:
heart["columns"]["trestbps"]

{'datatype': 'int64',
 'null_count': 0,
 'null_percentage': 0.0,
 'all_missing': False,
 'distinct_count': 50,
 'column_sample': [145, 160, 120, 130, 130],
 'statistics': {'min': 94.0,
  'max': 200.0,
  'mean': 131.69360269360268,
  'median': 130.0},
 'sql_profile': {'null_count': 0,
  'distinct_count': 50,
  'statistics': {'min': 94,
   'max': 200,
   'mean': 131.69360269360268,
   'median': 130.0}}}

In [9]:
cardio["columns"]["restingBP"]

{'datatype': 'int64',
 'null_count': 0,
 'null_percentage': 0.0,
 'all_missing': False,
 'distinct_count': 95,
 'column_sample': [171, 94, 133, 138, 199],
 'statistics': {'min': 94.0, 'max': 200.0, 'mean': 151.747, 'median': 147.0},
 'sql_profile': {'null_count': 0,
  'distinct_count': 95,
  'statistics': {'min': 94, 'max': 200, 'mean': 151.747, 'median': 147.0}}}

### Checking all columns have metadata

In [10]:
for df in [heart, cardio]:
    for column in df["columns"]:
        print(column, df["columns"][column].keys())

patient_id dict_keys(['datatype', 'null_count', 'null_percentage', 'all_missing', 'distinct_count', 'column_sample', 'statistics', 'sql_profile'])
age dict_keys(['datatype', 'null_count', 'null_percentage', 'all_missing', 'distinct_count', 'column_sample', 'statistics', 'sql_profile'])
sex dict_keys(['datatype', 'null_count', 'null_percentage', 'all_missing', 'distinct_count', 'column_sample', 'statistics', 'sql_profile'])
cp dict_keys(['datatype', 'null_count', 'null_percentage', 'all_missing', 'distinct_count', 'column_sample', 'statistics', 'sql_profile'])
trestbps dict_keys(['datatype', 'null_count', 'null_percentage', 'all_missing', 'distinct_count', 'column_sample', 'statistics', 'sql_profile'])
chol dict_keys(['datatype', 'null_count', 'null_percentage', 'all_missing', 'distinct_count', 'column_sample', 'statistics', 'sql_profile'])
fbs dict_keys(['datatype', 'null_count', 'null_percentage', 'all_missing', 'distinct_count', 'column_sample', 'statistics', 'sql_profile'])
restecg 

### Testing data profiling function

In [11]:
heart_df = pd.read_csv(heart_path)
cardio_df = pd.read_csv(cardio_path)

heart_sql = sql_profile(heart_df)
cardio_sql = sql_profile(cardio_df)

In [12]:
heart_sql["trestbps"]

{'null_count': 0,
 'distinct_count': 50,
 'statistics': {'min': 94,
  'max': 200,
  'mean': 131.69360269360268,
  'median': 130.0}}

In [13]:
cardio_sql["restingBP"]

{'null_count': 0,
 'distinct_count': 95,
 'statistics': {'min': 94, 'max': 200, 'mean': 151.747, 'median': 147.0}}

### Comparing extract_metadata (pandas) with sql_profile (SQL)

In [14]:
#null counts
x = heart["columns"]["trestbps"]["null_count"] == heart_sql["trestbps"]["null_count"]

#distinct counts
y = heart["columns"]["trestbps"]["distinct_count"] == heart_sql["trestbps"]["distinct_count"]

#numerical mean
z = abs(heart["columns"]["trestbps"]["statistics"]["mean"] - heart_sql["trestbps"]["statistics"]["mean"]) < 0.000001

if (x == y == z == True):
    print("Match!")

Match!


### Checking SQL profile within extract_metadata function

In [15]:
heart = extract_metadata(heart_path)

heart["columns"]["trestbps"]["sql_profile"]

#note: these will match the pandas metadata as tested in the previous cell.

{'null_count': 0,
 'distinct_count': 50,
 'statistics': {'min': 94,
  'max': 200,
  'mean': 131.69360269360268,
  'median': 130.0}}

### Testing primary_keys function

In [16]:
print("Heart primary key candidates:", primary_keys(heart_df))
print("Cardio primary key candidates:", primary_keys(cardio_df))

Heart primary key candidates: ['patient_id']
Cardio primary key candidates: ['patientid']


### Testing possible_relationships function

In [17]:
relationships = possible_relationships(heart_df, cardio_df)

for column, info in relationships.items():
    print(column, info)

age {'shared_values': 41, 'dataset1_distinct': 41, 'dataset2_distinct': 61}
oldpeak {'shared_values': 40, 'dataset1_distinct': 40, 'dataset2_distinct': 63}
slope {'shared_values': 3, 'dataset1_distinct': 3, 'dataset2_distinct': 4}


### Checking candidate primary keys within extract_metadata function

In [18]:
heart = extract_metadata("../datasets/cleaned/heart_disease_cleaned.csv")
cardio = extract_metadata("../datasets/cleaned/cardiovascular_cleaned.csv")

print(heart["potential_keys"])
print(cardio["potential_keys"])

#matches the previous testing cell!

['patient_id']
['patientid']


### Exporting metadata into JSON

In [19]:
import json

export_metadata(heart, "../outputs/metadata/heart_metadata.json")
export_metadata(cardio, "../outputs/metadata/cardio_metadata.json")

with open("../outputs/metadata/heart_metadata.json") as file:
    heart_json = json.load(file)

with open("../outputs/metadata/cardio_metadata.json") as file:
    cardio_json = json.load(file)

k = heart_json.keys() == cardio_json.keys()
hr = heart_json["row_count"] == heart["row_count"]
cr = cardio_json["row_count"] == cardio["row_count"]

if (k == hr == cr == True):
    print("Correct!")

Metadata exported to ../outputs/metadata/heart_metadata.json
Metadata exported to ../outputs/metadata/cardio_metadata.json
Correct!


### Final Validation

In [20]:
# Used AI tools to help with final validation code

required = [
    "datatype", "null_count", "null_percentage",
    "all_missing", "distinct_count", "column_sample",
    "sql_profile"
]

for dataset in [heart, cardio]:
    assert len(dataset["columns"]) == dataset["column_count"]

    for column, info in dataset["columns"].items():
        for field in required:
            assert field in info, f"Missing {field} in {column}"

print("All columns contain required metadata!")

All columns contain required metadata!


In [21]:
#comparing SQL and pandas profiling
for dataset in [heart, cardio]:
    for column, info in dataset["columns"].items():
        #getting the SQL metadata
        sql = info["sql_profile"]

        #checking if they match with no error
        assert info["null_count"] == sql["null_count"]
        assert info["distinct_count"] == sql["distinct_count"]

        #summary statistics for numerical columns
        if "statistics" in info:
            for stat in ["min", "max", "mean", "median"]:
                pandas_val = info["statistics"][stat]
                sql_val = sql["statistics"][stat]

                if pandas_val is None:
                    assert sql_val is None
                else:
                    assert abs(pandas_val - sql_val) < 0.000001

print("All SQL and pandas statistics match!")


All SQL and pandas statistics match!


### Final Summary 

#### Output Metadata:
* Dataset Metadata: row and column counts, column names, data types
* Column Metadata: null counts, distinct counts, sample values, numerical summary stats, categorical frequencies
* SQL profiling using DuckDB: null counts, distinct counts, numerical summary statistics
* Potential keys: possible primary keys for a dataset
* Relationships: shared column names and value overlap counts


## Testing Imaging: extract_metadata_img

In [22]:
#extract metadata for imaging dataset
metadata = extract_metadata(image_path)

print("Dataset:", metadata["dataset"])
print("Modality:", metadata["modality"])

#counts
print("\nDataset Information:")
print("Samples:", metadata["sample_count"])
print("ROI files:", metadata["roi_count"])
print("Orientation records:", metadata["orientation_count"])
print("Masks:", metadata["mask_count"])
print("Native images:", metadata["native_image_count"])

Dataset: ventricular_tissue_imaging
Modality: microscopy_imaging

Dataset Information:
Samples: 22
ROI files: 22
Orientation records: 22
Masks: 13
Native images: 0


### Inspecting one sample

In [23]:
id = "191007_1_overview"
sample = metadata["samples"][id]

print(json.dumps(sample, indent=4))

{
    "sample_id": "191007_1_overview",
    "files": {
        "roi": {
            "filename": "191007_1_overview.txt",
            "path": "/Users/rakshanpatnaik/Desktop/D37-Semantic-Data-Lake/datasets/ventricular data/ROIvalues/191007_1_overview.txt",
            "extension": ".txt",
            "filesize": 40
        },
        "orientation": {
            "filename": "orientationStack.txt",
            "path": "/Users/rakshanpatnaik/Desktop/D37-Semantic-Data-Lake/datasets/ventricular data/orientationStack.txt",
            "extension": ".txt",
            "filesize": 280
        },
        "mask": {
            "filename": "191007_1_Mask.tif",
            "path": "/Users/rakshanpatnaik/Desktop/D37-Semantic-Data-Lake/datasets/ventricular data/Masks/191007_1_Mask.tif",
            "extension": ".tif",
            "filesize": 6971360
        }
    },
    "quality_flags": [
        "mask_uses_0_255_encoding"
    ],
    "roi": {
        "coordinates": [
            [
                10

### Checking overall metadata

In [24]:
#converting into summary table for each imaging sample
samp_rows = []

for id, sample in metadata["samples"].items():
    roi = sample.get("roi", {})
    orientation = sample.get("orientation", {})
    mask = sample.get("mask", {})
    relationships = sample.get("relationships", {})

    samp_rows.append({
        "sample_id": id,
        "included_in_study": sample.get("included_in_study"),
        "has_roi": relationships.get("has_roi"),
        "has_orientation": relationships.get("has_orientation"),
        "has_mask": relationships.get("has_mask"),
        "has_native_image": relationships.get("has_native_image"),
        "roi_width_px": roi.get("width_px"),
        "roi_height_px": roi.get("height_px"),
        "rotation_angle": orientation.get("rotation_angle"),
        "mask_nonzero_percentage": mask.get("nonzero_percentage"),
        "mask_dimension_match": mask.get("roi_dimension_match"),
        "quality_flags": ", ".join(sample.get("quality_flags", []))
    })

df = pd.DataFrame(samp_rows)

df

,sample_id,included_in_study,has_roi,has_orientation,has_mask,has_native_image,roi_width_px,roi_height_px,rotation_angle,mask_nonzero_percentage,mask_dimension_match,quality_flags
0,190913_1_overview,False,True,True,False,False,664.0,664.0,0.0,NaN,None,
1,190930_1_overview,True,True,True,True,False,4628.0,1583.0,0.0,55.10,True,
2,191007_1_overview,True,True,True,True,False,2640.0,2640.0,118.0,8.79,True,mask_uses_0_255_encoding
3,191007_2_overview,True,True,True,False,False,3434.0,1980.0,40.0,NaN,None,
4,191017_1_overview,True,True,True,True,False,2644.0,2648.0,125.0,6.87,True,roi_preliminary_validation_failed
5,191017_5_overview,True,True,True,True,False,2640.0,2641.0,121.0,10.96,True,
6,191101_1_overview,True,True,True,True,False,1321.0,4882.0,20.0,5.45,True,
7,191107_2_overview,True,True,True,False,False,2331.0,1936.0,10.0,NaN,None,
8,191108_1_overview,True,True,True,True,False,2653.0,2648.0,90.0,36.79,True,
9,191114_3_overview,True,True,True,False,False,2640.0,2640.0,112.0,NaN,None,


### Checking values above with EDA (manual validation)

In [25]:
assert metadata["sample_count"] == 22
assert metadata["roi_count"] == 22
assert metadata["orientation_count"] == 22
assert metadata["mask_count"] == 13

#no error
print("no error")

no error


In [26]:
#checking excluded sample found in EDA
assert metadata["samples"]["190913_1_overview"]["included_in_study"] == False

#checking for mask encoding discrepancy
sample = metadata["samples"]["191007_1_overview"]
assert "mask_uses_0_255_encoding" in sample["quality_flags"]

#checking mask dimensions
matching_masks = [
    sample for sample in metadata["samples"].values()
    if "mask" in sample
    and sample["mask"].get("roi_dimension_match") == True
]

assert len(matching_masks) == 13

#no error
print("no error")

no error


### Exporting imaging metadata into JSON

In [27]:
metadata

{'dataset': 'ventricular_tissue_imaging',
 'source_path': '/Users/rakshanpatnaik/Desktop/D37-Semantic-Data-Lake/datasets/ventricular data',
 'modality': 'microscopy_imaging',
 'sample_count': 22,
 'roi_count': 22,
 'orientation_count': 22,
 'mask_count': 13,
 'native_image_count': 0,
 'samples': {'190913_1_overview': {'sample_id': '190913_1_overview',
   'files': {'roi': {'filename': '190913_1_overview.txt',
     'path': '/Users/rakshanpatnaik/Desktop/D37-Semantic-Data-Lake/datasets/ventricular data/ROIvalues/190913_1_overview.txt',
     'extension': '.txt',
     'filesize': 40},
    'orientation': {'filename': 'orientationStack.txt',
     'path': '/Users/rakshanpatnaik/Desktop/D37-Semantic-Data-Lake/datasets/ventricular data/orientationStack.txt',
     'extension': '.txt',
     'filesize': 280}},
   'quality_flags': [],
   'roi': {'coordinates': [[2979.0, 1524.0],
     [3643.0, 1524.0],
     [3643.0, 2188.0],
     [2979.0, 2188.0]],
    'point_count': 4,
    'x_min': 2979.0,
    'x_ma

In [28]:
export_metadata(metadata, f"{repo_root}/outputs/metadata/ventricular_imaging_metadata.json")

print("Imaging metadata exported.")

Metadata exported to /Users/rakshanpatnaik/Desktop/D37-Semantic-Data-Lake/outputs/metadata/ventricular_imaging_metadata.json
Imaging metadata exported.
